In [1]:
!pip install sqlalchemy

In [2]:
!pip install pymysql

In [3]:
pip show sqlalchemy | grep Version

Version: 2.0.51
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Establish a connection between Python and the Sakila database.

import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine
import getpass

In [5]:
password = getpass.getpass()

connection_string = 'mysql+pymysql://root:' + password + '@localhost/sakila'
engine = create_engine(connection_string)

In [8]:
# Write a Python function called rentals_month that retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame. 
# The function should take in three parameters:
    # engine
    # month
    # year

def rentals_month(engine, month, year):
    query = f"""
        SELECT * FROM rental
        WHERE MONTH(rental_date) = {month} 
        AND YEAR(rental_date) = {year}
    """
    return pd.read_sql(query, engine)

# Call the function
rentals_month(engine, 5, 2005)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53
...,...,...,...,...,...,...,...
1151,1153,2005-05-31 21:36:44,2725,506,2005-06-10 01:26:44,2,2006-02-15 21:30:53
1152,1154,2005-05-31 21:42:09,2732,59,2005-06-08 16:40:09,1,2006-02-15 21:30:53
1153,1155,2005-05-31 22:17:11,2048,251,2005-06-04 20:27:11,2,2006-02-15 21:30:53
1154,1156,2005-05-31 22:37:34,460,106,2005-06-01 23:02:34,2,2006-02-15 21:30:53


In [9]:
def rental_count_month(df, month, year):
    df_grouped = df.groupby("customer_id")["rental_id"].count().reset_index()
    df_grouped = df_grouped.rename(columns={"rental_id": f"rentals_{month:02d}_{year}"})
    return df_grouped

In [10]:
may_2005 = rentals_month(engine, 5, 2005)
rental_count_month(may_2005, 5, 2005)

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3
...,...,...
515,594,4
516,595,1
517,596,6
518,597,2


In [11]:
def compare_rentals(df1, df2):
    merged_df = df1.merge(df2, on="customer_id", how="outer")
    merged_df = merged_df.fillna(0)
    cols = [col for col in merged_df.columns if col != "customer_id"]
    merged_df["difference"] = merged_df[cols[0]] - merged_df[cols[1]]
    return merged_df

# Call all functions
may_2005 = rentals_month(engine, 5, 2005)
june_2005 = rentals_month(engine, 6, 2005)

may_count = rental_count_month(may_2005, 5, 2005)
june_count = rental_count_month(june_2005, 6, 2005)

result = compare_rentals(may_count, june_count)
result

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2.0,7.0,-5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,-2.0
3,4,0.0,6.0,-6.0
4,5,3.0,5.0,-2.0
...,...,...,...,...
593,595,1.0,2.0,-1.0
594,596,6.0,2.0,4.0
595,597,2.0,3.0,-1.0
596,598,0.0,1.0,-1.0
